# Minería de texto en español: clasificación de solicitudes de clientes

## Objetivo

Una empresa recibe mensajes de clientes y quiere asignarlos automáticamente a un área: facturación, soporte técnico, cancelación o entrega. En este notebook construiremos un clasificador de texto reproducible usando TF-IDF y regresión logística.

El flujo cubre: creación/carga del dataset, exploración, limpieza, vectorización, entrenamiento, evaluación, interpretación de términos y generación de conclusiones.

## ¿Qué problema de negocio estamos resolviendo?

Cuando una empresa recibe cientos o miles de mensajes, una persona debe leer cada solicitud y enviarla al área correcta. Ese proceso puede generar tres problemas: tiempos de respuesta altos, asignaciones equivocadas y trabajo repetitivo.

La minería de texto permite convertir mensajes libres en información estructurada. En este caso, el resultado esperado no es redactar una respuesta automática, sino **predecir el área responsable**. Esto puede utilizarse para:

- enviar un ticket directamente a facturación, soporte, cancelaciones o logística;
- priorizar mensajes urgentes o de baja confianza;
- medir qué temas generan más demanda;
- reducir reasignaciones y tiempos de atención.

Este ejemplo es educativo: el dataset es pequeño y artificial. Su utilidad principal es mostrar el razonamiento completo que después se aplicaría a mensajes reales.

## ¿Qué aprenderá al ejecutarlo?

Al finalizar podrá explicar la ruta completa:

1. Un texto humano se convierte en variables numéricas.
2. El algoritmo aprende patrones a partir de ejemplos etiquetados.
3. El modelo se prueba con mensajes que no vio durante el entrenamiento.
4. Las métricas indican si las predicciones son suficientemente confiables.
5. Los errores y términos relevantes ayudan a decidir cómo mejorar el proceso.

La idea clave es que un modelo no memoriza solamente palabras: aprende asociaciones estadísticas entre expresiones y categorías.

## 1. Preparación del entorno

Colab ya incluye las bibliotecas utilizadas. Si ejecuta este notebook fuera de Colab, instale pandas, scikit-learn, matplotlib y seaborn. Las semillas aleatorias permiten repetir el experimento.

In [ ]:
import io
import re
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 120)

## 2. Dataset autocontenido

Para que el ejemplo sea ejecutable sin descargar archivos, el dataset se define como CSV dentro del notebook. El mismo contenido se guarda como solicitudes_clientes.csv, por lo que puede reemplazarse posteriormente por datos reales.

Cada registro contiene un mensaje y su categoría correcta. En un proyecto real, estas etiquetas deben ser revisadas por personal del negocio.

### ¿Por qué necesitamos una categoría?

La columna categoria es la respuesta correcta que usaremos para enseñar al modelo. Por ejemplo, el mensaje “el paquete aún no llega” está etiquetado como entrega. El modelo observa muchos ejemplos y aprende que palabras como paquete, envío, guía y entrega suelen estar relacionadas.

La calidad de esta columna es crítica: si los ejemplos están mal clasificados, el algoritmo aprenderá reglas equivocadas. En un proyecto real conviene definir un manual de etiquetado y medir el acuerdo entre varias personas.

In [ ]:
csv_text = '''texto,categoria
No reconozco el cargo de mi factura,facturacion
Necesito una copia de la factura del mes,facturacion
Me cobraron dos veces el mismo servicio,facturacion
Quiero corregir mis datos fiscales,facturacion
El importe de la factura no coincide,facturacion
¿Cuándo se genera mi próxima factura?,facturacion
Solicito factura con mi RFC,facturacion
Tengo un cobro pendiente que no entiendo,facturacion
El pago aparece rechazado aunque tengo saldo,facturacion
Necesito aclarar un cargo en mi cuenta,facturacion
¿Pueden enviarme el comprobante de pago?,facturacion
La factura tiene un error en el domicilio,facturacion
La aplicación se cierra al iniciar sesión,soporte_tecnico
No puedo ingresar a mi cuenta,soporte_tecnico
La página muestra un error al guardar,soporte_tecnico
El sistema está muy lento desde esta mañana,soporte_tecnico
No recibo el código de verificación,soporte_tecnico
La contraseña temporal no funciona,soporte_tecnico
El botón de pago no responde,soporte_tecnico
La aplicación se queda cargando,soporte_tecnico
Mi usuario quedó bloqueado,soporte_tecnico
No puedo actualizar la aplicación,soporte_tecnico
El portal marca un error desconocido,soporte_tecnico
La notificación no aparece en el sistema,soporte_tecnico
Quiero cancelar mi suscripción,cancelacion
Solicito dar de baja el servicio,cancelacion
Ya no deseo continuar con el plan,cancelacion
¿Cómo puedo cancelar mi cuenta?,cancelacion
Necesito terminar el contrato,cancelacion
Quiero eliminar mi suscripción mensual,cancelacion
Deseo cancelar antes del siguiente cobro,cancelacion
Por favor procesen la baja de mi servicio,cancelacion
Quiero cerrar definitivamente mi cuenta,cancelacion
Solicito la cancelación inmediata,cancelacion
Ya no necesito el plan contratado,cancelacion
Necesito confirmar que mi cuenta fue cancelada,cancelacion
¿Dónde está mi pedido?,entrega
El paquete aún no llega,entrega
Quiero conocer la fecha de entrega,entrega
Mi envío aparece detenido,entrega
El pedido llegó incompleto,entrega
Recibí un producto dañado en la entrega,entrega
Necesito cambiar la dirección de envío,entrega
El repartidor no pudo localizar mi domicilio,entrega
¿Me pueden compartir la guía del paquete?,entrega
La entrega está retrasada varios días,entrega
No recibí el paquete marcado como entregado,entrega
Quiero rastrear mi envío,entrega'''

df = pd.read_csv(io.StringIO(csv_text))
df.to_csv('solicitudes_clientes.csv', index=False, encoding='utf-8')
print(f'Registros: {len(df)} | Categorías: {df.categoria.nunique()}')
display(df.head())

Interpretación: el dataset está balanceado; cada categoría contiene 12 mensajes. Esto facilita el aprendizaje, aunque un proyecto real suele presentar desbalance entre áreas.

### ¿Qué debemos observar en la exploración?

Antes de entrenar conviene responder: ¿hay suficientes ejemplos por clase?, ¿hay categorías casi vacías?, ¿existen duplicados?, ¿hay textos demasiado cortos?, ¿se está usando una misma plantilla que podría hacer que el modelo parezca mejor de lo que realmente es?

La gráfica siguiente no evalúa todavía al modelo. Sirve para revisar si el problema está razonablemente representado.

In [ ]:
display(df['categoria'].value_counts().to_frame('cantidad'))
sns.countplot(data=df, x='categoria', order=df['categoria'].value_counts().index)
plt.title('Distribución de solicitudes por categoría')
plt.xticks(rotation=20)
plt.show()

## 3. Limpieza del texto

La función siguiente normaliza minúsculas, elimina acentos, URLs, signos y espacios repetidos. No se eliminan todas las palabras vacías automáticamente: en español, algunas palabras cortas pueden aportar contexto. La limpieza debe adaptarse al dominio.

El objetivo no es borrar la mayor cantidad de texto posible. Es reducir variaciones irrelevantes sin eliminar señales importantes. Por ejemplo, puede ser útil conservar números de pedido, montos o palabras como no, nunca y cancelación, dependiendo del caso.

In [ ]:
def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'https?://\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'[^a-zñáéíóúü0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

df['texto_limpio'] = df['texto'].apply(limpiar_texto)
display(df[['texto', 'texto_limpio']].head(8))

## 4. Separación de entrenamiento y prueba

El modelo aprende con el 75% de los mensajes y se evalúa con el 25% restante. stratify conserva la proporción de categorías en ambos grupos. La separación ocurre antes de ajustar el vectorizador para evitar fuga de información.

### ¿Por qué no entrenamos con todos los datos?

Si evaluamos con los mismos mensajes que usamos para entrenar, no sabremos si el modelo generaliza. El conjunto de prueba simula mensajes futuros.

La fuga de información ocurre cuando una etapa aprende de la prueba antes de evaluar, por ejemplo, calculando TF-IDF con todo el dataset. Pipeline evita ese error al ajustar la transformación únicamente con X_train.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['texto_limpio'], df['categoria'], test_size=0.25,
    random_state=RANDOM_STATE, stratify=df['categoria']
)
print(f'Entrenamiento: {len(X_train)} mensajes')
print(f'Prueba: {len(X_test)} mensajes')

## 5. Modelo TF-IDF + regresión logística

TF-IDF convierte cada texto en números: aumenta el peso de palabras importantes dentro de un documento y reduce el peso de palabras demasiado comunes. La regresión logística aprende qué combinaciones de palabras distinguen cada categoría.

Se incluyen unigramas y bigramas para capturar expresiones como dar baja o fecha entrega. Pipeline garantiza que el vectorizador se ajuste solamente con entrenamiento.

### ¿Qué significa TF-IDF?

- TF mide qué tan frecuente es un término dentro de un mensaje.
- IDF reduce el peso de términos que aparecen en casi todos los mensajes.
- El producto TF-IDF deja representadas las palabras distintivas.

Un unigrama es una palabra individual, como factura. Un bigrama es una combinación de dos palabras, como dar baja. Los bigramas aportan contexto, pero también aumentan el número de variables.

In [ ]:
modelo = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)),
    ('clasificador', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)
print(f'Exactitud en prueba: {accuracy_score(y_test, predicciones):.2%}')
print(classification_report(y_test, predicciones, zero_division=0))

Cómo leer las métricas: precision indica cuántos mensajes asignados a una clase eran correctos; recall indica cuántos mensajes reales de esa clase fueron encontrados; f1-score combina ambos. En proyectos operativos, la métrica prioritaria depende del costo de equivocarse.

### Cómo convertir las métricas en una decisión

Si el objetivo es no enviar tickets al área incorrecta, se prioriza precision. Si el objetivo es no dejar solicitudes sin detectar, se prioriza recall. F1 es útil cuando ambos errores importan de forma similar.

No debe interpretarse una exactitud alta como garantía de producción cuando el dataset es pequeño o muy sencillo. La pregunta de negocio es: ¿el costo de automatizar los casos claros es menor que el costo de revisarlos manualmente?

In [ ]:
etiquetas = sorted(df['categoria'].unique())
matriz = confusion_matrix(y_test, predicciones, labels=etiquetas)
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', xticklabels=etiquetas, yticklabels=etiquetas)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de confusión')
plt.show()

errores = pd.DataFrame({'texto': X_test, 'real': y_test, 'predicha': predicciones})
display(errores[errores.real != errores.predicha])

## 6. Interpretación del modelo

Los coeficientes positivos muestran términos que empujan la decisión hacia una categoría. Esto permite explicar el resultado a las áreas de negocio y detectar palabras ambiguas o etiquetas de baja calidad.

### ¿Cómo interpretar los términos?

Si aparecen factura, cobro o RFC para facturacion, el modelo está usando señales coherentes con el negocio. Si aparecen términos accidentales, como nombres de personas, fechas o frases repetidas de una sola fuente, puede existir un atajo o sesgo.

La interpretabilidad no sustituye la evaluación. Sirve para formular preguntas: ¿las palabras relevantes tienen sentido?, ¿falta vocabulario?, ¿dos categorías usan expresiones demasiado parecidas?

In [ ]:
vectorizador = modelo.named_steps['tfidf']
clasificador = modelo.named_steps['clasificador']
terminos = vectorizador.get_feature_names_out()

for indice, clase in enumerate(clasificador.classes_):
    pesos = clasificador.coef_[indice]
    mejores = pesos.argsort()[-10:][::-1]
    print(f'\n{clase}: ' + ', '.join(terminos[mejores]))

## 7. Predicción de nuevos mensajes

Usamos el modelo como si llegaran solicitudes nuevas. La probabilidad permite identificar casos dudosos para revisión humana.

In [ ]:
nuevos_mensajes = [
    'No entiendo por qué mi pago fue cobrado dos veces',
    'El paquete sigue sin llegar y necesito rastrearlo',
    'Quiero dar de baja mi plan antes del próximo cobro',
    'La aplicación no me deja iniciar sesión'
]

textos_nuevos = [limpiar_texto(x) for x in nuevos_mensajes]
pred = modelo.predict(textos_nuevos)
probs = modelo.predict_proba(textos_nuevos)
resultado = pd.DataFrame({
    'mensaje': nuevos_mensajes,
    'categoria_predicha': pred,
    'confianza': probs.max(axis=1).round(3)
})
display(resultado)

### Posible uso operativo

Una regla sencilla sería automatizar únicamente los mensajes cuya confianza supere un umbral, por ejemplo 0.80, y enviar los demás a una bandeja de revisión. El umbral no debe elegirse arbitrariamente: se calibra con datos reales y con el costo de cada tipo de error.

Ejemplo de flujo: mensaje recibido → limpieza → predicción → si la confianza es alta, asignación automática → si es baja, revisión humana → nueva etiqueta incorporada para mejorar el modelo.

## 8. Limitaciones y controles necesarios

Este notebook no debe conectarse directamente a un proceso crítico sin controles adicionales. Antes de usar un modelo con datos reales, revise:

- tamaño y representatividad del dataset;
- duplicados, mensajes vacíos y etiquetas inconsistentes;
- desempeño por categoría y no solo promedio general;
- cambios de vocabulario a lo largo del tiempo;
- información personal, confidencial o sensible;
- mecanismo para corregir predicciones y reentrenar;
- registro de versiones del modelo y de los datos.

La automatización recomendada es gradual: primero apoyar al agente humano, después automatizar únicamente los casos de alta confianza y finalmente ampliar el alcance con evidencia.

## 9. Conclusiones

- TF-IDF ofrece una línea base rápida, interpretable y adecuada para mensajes cortos con vocabulario relativamente claro.
- La matriz de confusión muestra qué áreas podrían confundirse; esos errores son más útiles que observar únicamente la exactitud.
- Los términos relevantes ayudan a validar si el modelo está aprendiendo señales razonables o atajos indeseados.
- La confianza puede utilizarse para enrutar automáticamente los casos claros y enviar los casos dudosos a revisión humana.
- Antes de producción se requiere un dataset mayor y representativo, validación temporal, monitoreo de cambios en vocabulario y revisión de privacidad.

### Siguientes mejoras

1. Aumentar y balancear las etiquetas con ejemplos reales.
2. Comparar contra embeddings y un modelo transformer en español.
3. Ajustar un umbral de confianza según el costo de cada error.
4. Medir el impacto operativo: tiempo de respuesta, reasignaciones y satisfacción del cliente.

### Conclusión ejecutiva

La minería de texto puede convertir una bandeja de mensajes en un flujo operativo medible. TF-IDF y regresión logística son una primera solución recomendable porque son rápidos, económicos y explicables. La decisión de llevarlos a producción debe basarse en datos reales, métricas por categoría, revisión humana y evidencia de que reducen tiempos o errores.